# 🏥 IMO Coding Intelligence API — Notebook 2: Upload Codes & Validate

This notebook shows how to **load medical codes** (manually or from a CSV) and **validate them** against the IMO Coding Intelligence API.

The API checks each code against IMO's curated administrative coding assistance sets and flags:
- **Non-primary codes** — diagnosis codes that cannot be the primary diagnosis on a claim
- **Unspecified codes** — codes that lack required laterality, specificity, or detail

---

### What you'll cover
1. Re-authenticate and configure credentials
2. Define codes manually as Python dicts
3. Load codes from a CSV file using pandas
4. POST codes to the IMO Coding Intelligence API (`imo-admin-coding-sets`)
5. Parse and display validation results

> 💡 **Prerequisites:** Complete `01_Setup_and_Authentication.ipynb` first to understand the auth flow.

## Step 1: Setup — Imports & Authentication

In [ ]:
import requests
import json
import os
import pandas as pd

# ── Credentials ───────────────────────────────────────────────
CLIENT_ID     = os.environ.get("IMO_CODING_INTEL_CLIENT_ID",     "YOUR_CLIENT_ID_HERE")
CLIENT_SECRET = os.environ.get("IMO_CODING_INTEL_CLIENT_SECRET", "YOUR_CLIENT_SECRET_HERE")

AUTH_URL         = "https://api.imohealth.com/oauth/token"
CODING_INTEL_URL = "https://api.imohealth.com/codingintelligence/v1/rules/imo-admin-coding-sets"

# ── Get token ─────────────────────────────────────────────────
def get_access_token(client_id, client_secret):
    payload = {
        "grant_type":    "client_credentials",
        "client_id":     client_id,
        "client_secret": client_secret,
        "audience":      "https://api.imohealth.com"
    }
    r = requests.post(AUTH_URL, json=payload, timeout=30)
    r.raise_for_status()
    return r.json()["access_token"]

ACCESS_TOKEN = get_access_token(CLIENT_ID, CLIENT_SECRET)
HEADERS = {
    "Content-Type":  "application/json",
    "Authorization": f"Bearer {ACCESS_TOKEN}"
}
print("✓ Authenticated — ready to validate codes")

## Step 2: Define Codes Manually

Each code is a dict with two required fields:
- `code` — the actual code string (e.g. `"E11.9"`)
- `code_system` — the terminology system (e.g. `"ICD-10-CM"`)

The list below includes a mix of clean codes and intentionally problematic ones so you can see the validation results.

In [ ]:
# Each entry requires 'code' and 'code_system'
sample_codes = [
    {"code": "I10",      "code_system": "ICD-10-CM"},   # Essential hypertension — clean
    {"code": "E11.9",    "code_system": "ICD-10-CM"},   # Type 2 diabetes, unspecified — may flag unspecified
    {"code": "Z3A.10",   "code_system": "ICD-10-CM"},   # 10 weeks gestation — non-primary flag expected
    {"code": "M79.646",  "code_system": "ICD-10-CM"},   # Pain in unspecified finger — unspecified laterality
    {"code": "J18.9",    "code_system": "ICD-10-CM"},   # Pneumonia, unspecified — clean
]

print(f"✓ Loaded {len(sample_codes)} codes manually")
for c in sample_codes:
    print(f"   {c['code_system']:12s}  {c['code']}")

## Step 3: (Alternative) Load Codes from a CSV File

If you have codes in a CSV, use pandas to load them. Your CSV should have at minimum a `code` column and a `code_system` column (or you can hardcode the system if all codes are from the same terminology).

In [ ]:
CSV_PATH = "path/to/your/codes.csv"   # ← update this path

# Expected CSV columns: code, code_system (or just code if all ICD-10-CM)
# Example CSV:
#   code,code_system
#   I10,ICD-10-CM
#   E11.9,ICD-10-CM

import os
if os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH)

    # If there's no code_system column, default to ICD-10-CM
    if "code_system" not in df.columns:
        df["code_system"] = "ICD-10-CM"

    csv_codes = df[["code", "code_system"]].dropna().to_dict(orient="records")
    print(f"✓ Loaded {len(csv_codes)} codes from CSV")
    print(df.head())
else:
    print("⚠️  CSV file not found — using manually defined sample_codes instead")
    csv_codes = sample_codes

# Use whichever source you prefer:
codes_to_validate = sample_codes   # swap to csv_codes if using CSV
print(f"\nTotal codes to validate: {len(codes_to_validate)}")

## Step 4: Call the Coding Intelligence API

We POST the code list to the `imo-admin-coding-sets` endpoint. The API checks each code against IMO's administrative value sets and returns per-code results with:

| Field | Meaning |
|---|---|
| `code` | The submitted code |
| `description` | IMO's description for the code |
| `is_nonprimary` | `true` if this code cannot be the primary diagnosis |
| `is_unspecified` | `true` if the code lacks required specificity/laterality |
| `message_text` | Human-readable explanation of any issue |

In [ ]:
def validate_billing_codes(codes: list, headers: dict) -> list:
    """
    POST codes to the IMO imo-admin-coding-sets endpoint.

    Args:
        codes:   List of {"code": "...", "code_system": "..."} dicts
        headers: Auth headers from get_access_token()

    Returns:
        List of per-code validation result dicts
    """
    payload = {
        "library_title": "IMO Precision Administrative Coding Assistance Sets",
        "category_titles": [],
        "value_set_ids": [
            8054, 11248, 11247, 8057, 10098, 10090, 10095, 10093,
            10094, 10096, 10101, 10087, 10092, 10089, 10102, 10088,
            10097, 10099
        ],
        "codes": codes
    }

    print(f"Validating {len(codes)} code(s) against IMO Coding Intelligence API...")
    response = requests.post(CODING_INTEL_URL, headers=headers, json=payload, timeout=30)
    response.raise_for_status()

    result = response.json()
    validated = result.get("codes", [])
    print(f"✓ API response received — {len(validated)} result(s) returned")
    return validated


validated_results = validate_billing_codes(codes_to_validate, HEADERS)

## Step 5: Display Results

Parse the API response and display a clean summary table showing which codes passed validation and which were flagged.

In [ ]:
# ── Console-style detailed output ────────────────────────────
print("=" * 70)
print("CODING INTELLIGENCE VALIDATION RESULTS")
print("=" * 70)

flagged = 0
for r in validated_results:
    code        = r.get("code", "—")
    description = r.get("description", "No description")
    is_nonprim  = r.get("is_nonprimary", False)
    is_unspec   = r.get("is_unspecified", False)
    message     = r.get("message_text", "")

    if is_nonprim or is_unspec:
        flagged += 1
        flags = []
        if is_nonprim: flags.append("NON-PRIMARY")
        if is_unspec:  flags.append("UNSPECIFIED")
        print(f"\n  ⚠️  {code} — {description}")
        print(f"      Flags   : {', '.join(flags)}")
        print(f"      Reason  : {message}")
    else:
        print(f"\n  ✅  {code} — {description}  [Valid for billing]")

print(f"\n{'=' * 70}")
print(f"Summary: {flagged} of {len(validated_results)} code(s) flagged")
print("=" * 70)

In [ ]:
# ── Pandas summary table ─────────────────────────────────────
rows = []
for r in validated_results:
    flags = []
    if r.get("is_nonprimary"): flags.append("Non-Primary")
    if r.get("is_unspecified"): flags.append("Unspecified")
    rows.append({
        "Code":        r.get("code", ""),
        "Description": r.get("description", ""),
        "Status":      "⚠️ Review" if flags else "✅ Valid",
        "Flags":       ", ".join(flags) if flags else "—",
        "Message":     r.get("message_text", "—")
    })

df_results = pd.DataFrame(rows)
df_results

## Step 6: Check CMS Excludes 1 Conflicts

In addition to billing validation, the IMO API can detect **Excludes 1 conflicts** — pairs of ICD-10-CM codes that **cannot be billed together** on the same claim per CMS rules.

The `cms-excludes1` endpoint accepts the same code list (ICD-10-CM only) and returns any conflicting pairs found via `results[].note_codes[]`.

In [ ]:
EXCLUDES1_URL = "https://api.imohealth.com/codingintelligence/v1/rules/cms-excludes1"

def check_excludes1(codes: list, headers: dict) -> list:
    """
    POST ICD-10-CM codes to the cms-excludes1 endpoint.
    Each code requires a 'record_id' field per the API spec.

    Returns:
        List of result dicts; each has 'code' and 'note_codes' (conflicting codes)
    """
    # Filter to ICD-10-CM only — Excludes1 is an ICD-10-CM rule
    icd_codes = [c for c in codes if "ICD-10" in c.get("code_system", "").upper() or c.get("code_system", "").upper() == "ICD-10-CM"]

    if not icd_codes:
        print("No ICD-10-CM codes in the list — Excludes 1 check skipped.")
        return []

    payload = {
        "codes": [
            {
                "code":        c["code"],
                "code_system": c["code_system"],
                "record_id":   str(i + 1)
            }
            for i, c in enumerate(icd_codes)
        ]
    }

    print(f"Checking {len(icd_codes)} ICD-10-CM code(s) for Excludes 1 conflicts...")
    response = requests.post(EXCLUDES1_URL, headers=headers, json=payload, timeout=30)
    response.raise_for_status()

    data    = response.json()
    results = data.get("results", [])
    total   = data.get("total_results", 0)
    print(f"✓ Excludes 1 check complete — {total} conflict(s) found")
    return results


excludes1_results = check_excludes1(codes_to_validate, HEADERS)

In [ ]:
# ── Console conflict report ───────────────────────────────────
print("=" * 65)
print("CMS EXCLUDES 1 CONFLICT REPORT")
print("=" * 65)

if not excludes1_results:
    print("\n✅ No Excludes 1 conflicts detected — all codes are compatible.")
else:
    for r in excludes1_results:
        code       = r.get("code", "—")
        note_codes = r.get("note_codes", [])
        conflicting = [nc.get("code") for nc in note_codes if nc.get("code")]
        print(f"\n  ⛔ {code}  conflicts with: {', '.join(conflicting)}")
        print(f"     These codes cannot be billed together on the same claim.")

print(f"\n{'=' * 65}")

# ── Pandas conflict table ─────────────────────────────────────
conflict_rows = []
for r in excludes1_results:
    code = r.get("code", "—")
    for nc in r.get("note_codes", []):
        conflict_rows.append({
            "Code":           code,
            "Conflicts With": nc.get("code", "—"),
            "Rule":           "CMS Excludes 1 — Cannot code together"
        })

if conflict_rows:
    df_conflicts = pd.DataFrame(conflict_rows)
    display(df_conflicts)
else:
    print("✅ No conflicts to display")

## ✅ Summary

| Step | Description |
|------|-------------|
| 1 | Authenticated with IMO OAuth |
| 2 | Defined codes as Python dicts (or loaded from CSV) |
| 3 | POSTed codes to `imo-admin-coding-sets` endpoint |
| 4 | Parsed `is_nonprimary`, `is_unspecified`, and `message_text` per code |
| 5 | Displayed validation results in a pandas DataFrame |
| 6 | POSTed ICD-10-CM codes to `cms-excludes1` endpoint and displayed conflict pairs |

---

**This notebook covers the full coding intelligence validation pipeline in one place.**  
For a deeper dive into Excludes 1, see `03_Excludes1_Conflict_Check.ipynb`.